Import libraries ↓

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from keras.models import Sequential
from keras.layers import Dense, Input

Dataframe maken ↓

In [ ]:
df = pd.read_excel("../CSV/AmesHousing.xlsx")

One-hot encoding ↓

In [ ]:
categorische_feature = "House Style"

df_encoded = pd.get_dummies(df, columns=[categorische_feature])
print(df_encoded.head())

Features en target kiezen ↓

In [ ]:
x = df_encoded[["Gr Liv Area", "Overall Qual"] +
    [col for col in df_encoded.columns if col.startswith("House Style_")]]

y = df_encoded["SalePrice"]

print(x.shape)
print(y.shape)

x = x.astype(float)
y = y.astype(float)

Train en test split ↓

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.18, random_state=5)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

Neural network maken ↓

In [ ]:
model = Sequential()

#model.add(Input(shape=(10,)))
model.add(Input(shape=(x_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

Model trainen ↓

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

model.fit(x_train, y_train, epochs=55, batch_size=4, verbose=1, validation_split=0.1)

y_pred = model.predict(x_test)
print("Eerste 5 voorspellingen:")
print(y_pred[:5])

Evalueren ↓

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R²:", r2)

Testcase opslaan in Excel ↓

In [ ]:
import openpyxl
from openpyxl import load_workbook
from datetime import datetime
import os

EXCEL_BESTAND = 'testcases_regressie.xlsx'

# Lagen en neuronen ophalen uit het model
lagen_info = [f"{laag.units}" for laag in model.layers if hasattr(laag, 'units')]
lagen_str = ' → '.join(lagen_info)

# Features netjes als string
andere_features = "Gr Liv Area, Overall Qual"
target_feature = "SalePrice"

nieuwe_rij = [
    datetime.now().strftime('%Y-%m-%d %H:%M'),  # datum en tijd
    categorische_feature,  # categorische feature
    andere_features,  # andere features
    target_feature,  # target feature
    0.18,  # test size
    55,  # epochs
    4,  # batch size
    len(lagen_info),  # aantal lagen
    lagen_str,  # neuronen per laag
    round(mae, 2),  # MAE
    round(mse, 2),  # MSE
    round(r2, 4),  # R²
]

HEADER = [
    'Datum/tijd', 'Categorische feature', 'Andere features', 'Target feature',
    'Test size', 'Epochs', 'Batch size', 'Aantal lagen', 'Neuronen per laag',
    'MAE', 'MSE', 'R²'
]

if os.path.exists(EXCEL_BESTAND):
    wb = load_workbook(EXCEL_BESTAND)
    ws = wb.active
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Testcases'
    ws.append(HEADER)

ws.append(nieuwe_rij)
wb.save(EXCEL_BESTAND)

print(f'Testcase opgeslagen in: {os.path.abspath(EXCEL_BESTAND)}')